# Automata-02 — Expressions régulières : le théorème de Kleene, dans les deux sens

**Navigation** : [Index de la série](README.md) | [Automata-01](Automata-01-AutomatesFinis-Python.ipynb) | [Version C#](Automata-02-ExpressionsRegulieres-CSharp.ipynb)

> **Public : Découverte.** Deuxième carnet de la sous-série. Il reprend l'AFN à
> epsilon-transitions d'`Automata-01`, n'ajoute aucune dépendance (bibliothèque standard
> seule) et se sert de `re` comme **oracle externe**.

## Ce que ce notebook suppose

- savoir lire un AFD, un AFN, et ce qu'est une epsilon-transition (`Automata-01`, sections 1 à 3) ;
- Python : dictionnaires, ensembles, récursion.

## Ce qu'on y apprend

1. **Décrire** un langage par une expression régulière, puis convertir
   **expression → automate** par induction structurelle sur la syntaxe.
2. **Convertir** dans l'autre sens, **automate → expression**, en éliminant les états.
3. **Mesurer** ce que coûte chaque conversion, au lieu de l'affirmer.

**Vérification indépendante.** Les deux convertisseurs de ce carnet sont écrits ici, donc
ils pourraient se tromper ensemble. Chaque résultat est donc confronté au moteur de
regex de la bibliothèque standard (`re.fullmatch`), qui est une **implémentation
séparée**, sur **tous** les mots jusqu'à une longueur fixée.


## 1. Deux notations pour la même famille de langages

Une **expression régulière** se construit à partir de trois briques de base — l'ensemble
vide $\varnothing$, le mot vide $\varepsilon$, et une lettre $a$ de l'alphabet — et de
trois opérations : l'**union** $r \mid s$, la **concaténation** $rs$, et l'**étoile** $r^\star$.

Les trois opérations sont ordonnées par priorité : l'étoile lie le plus fort, la
concaténation ensuite, l'union le plus faible. C'est pour cela que `(a|b)*abb` se lit
« n'importe quel mot sur $\{a, b\}$, puis `abb` » et non « `a`, ou `b` répété, puis `abb` ».

Le manuel d'*Introduction to Automata Theory, Languages, and Computation*
(Hopcroft, Motwani & Ullman, 2001 ; copie archivée, chapitre 3) démontre l'équivalence
entre les deux notations en **deux théorèmes séparés** :

| Énoncé du manuel | Sens | Où |
|---|---|---|
| **Théorème 3.4** : si $L = L(A)$ pour un AFD $A$, il existe une expression régulière $R$ telle que $L = L(R)$ | automate → expression | p. 91, §3.2.1 |
| **Théorème 3.7** : tout langage défini par une expression régulière est défini par un automate fini | expression → automate | p. 102, §3.2.3 |

Copie archivée du manuel, chapitre 3 (PDF sans couche texte : les pages sont des images) :
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Automata Theory and Finite-State Methods\2001 - Introduction-to-automata-theory.pdf`

C'est leur **réunion** qu'on appelle le théorème de Kleene. Le manuel la démontre par
deux constructions distinctes, et ce carnet les implémente toutes les deux : la
section 2 suit le théorème 3.7, la section 3 suit le §3.2.2 (variante par élimination
d'états du théorème 3.4).

Pour manipuler les expressions de façon uniforme, on les représente comme un **arbre**
dont les feuilles sont $\varnothing$, $\varepsilon$ ou une lettre, et dont les nœuds
internes portent une opération. C'est exactement la structure sur laquelle l'induction
du théorème 3.7 travaille.

In [1]:
# Une expression reguliere est un ARBRE : None (langage vide) ou un tuple (operateur, enfants).
# Ce choix rend les operateurs composables et la taille mesurable, sans dependance externe.
from itertools import product

EPS = ("e",)      # epsilon : le langage reduit au mot vide
VIDE = None       # langage vide : aucun mot du tout


def lit(lettre: str) -> tuple:
    """Le langage {lettre}, pour une lettre de l'alphabet."""
    return ("l", lettre)


def union(x, y) -> tuple:
    """Le langage L(x) union L(y)."""
    return ("u", x, y)


def concat(x, y) -> tuple:
    """Le langage {u + v : u dans L(x), v dans L(y)}."""
    return ("c", x, y)


def etoile(x) -> tuple:
    """Le langage L(x)*, fermeture de Kleene."""
    return ("s", x)


def _entre(rx, contexte: str) -> str:
    """Parenthese un sous-arbre quand la lecture l'exige.

    Deux raisons d'ajouter des parentheses : l'operateur du sous-arbre est plus faible que
    celui du contexte (`a|b` dans une concatenation ou une etoile, `ab` dans une etoile) ; ou
    la meme etoile se repete, car une etoile n'est pas associative et `a**` n'est pas un motif
    valide.
    """
    if rx is None:
        return "(?!)"
    texte = rendre(rx)
    if rx[0] == "u" and contexte in ("c", "s"):
        return f"({texte})"
    if rx[0] == "c" and contexte == "s":
        return f"({texte})"
    if rx[0] == "s" and contexte == "s":
        return f"({texte})"
    return texte


def rendre(rx) -> str:
    """Rend l'AST en chaine lisible par re.fullmatch : parentheses seulement ou il en faut."""
    if rx is None:
        return "(?!)"          # assertion toujours fausse : ne matche aucun mot
    if rx[0] == "e":
        return ""              # epsilon s'ecrit comme la chaine vide
    if rx[0] == "l":
        return rx[1]
    if rx[0] == "u":
        return f"{_entre(rx[1], 'u')}|{_entre(rx[2], 'u')}"
    if rx[0] == "c":
        return f"{_entre(rx[1], 'c')}{_entre(rx[2], 'c')}"
    interieur = _entre(rx[1], "s")
    # L* d'un langage qui ne contient que le mot vide vaut {mot vide} : le motif vide dit cela,
    # alors que "" suivi de "*" ne compile pas ("nothing to repeat").
    return "" if interieur in ("", "(?!)") else f"{interieur}*"


def nb_noeuds(rx) -> int:
    """Nombre de noeuds de l'AST : la mesure qui predit la taille de l'automate de Thompson."""
    if rx is None or rx[0] in ("e", "l"):
        return 1
    return 1 + sum(nb_noeuds(sous) for sous in rx[1:])


# Trois expressions de reference, ecrites comme des arbres (et non comme des chaines) :
# AB designe le fragment a|b, reutilise dans les constructions qui suivent.
AB = union(lit("a"), lit("b"))
RX_ABB = concat(concat(concat(etoile(AB), lit("a")), lit("b")), lit("b"))
RX_ABC = concat(lit("a"), etoile(union(lit("b"), lit("c"))))
RX_AB_STAR = etoile(concat(lit("a"), lit("b")))

for nom, rx in (("(a|b)*abb", RX_ABB), ("a(b|c)*", RX_ABC), ("(ab)*", RX_AB_STAR)):
    print(f"{nom:<10} noeuds={nb_noeuds(rx):2d}  rendu={rendre(rx)!r}")


# Deux cas limites que ce rendu laissait passer : `etoile(EPS)` donnait "*" et
# `etoile(etoile(lit("a")))` donnait "a**", deux chaines que re refuse de compiler.
# Chaque temoin compile le motif rendu avec le moteur reel, puis verifie le langage accepte.
import re

for nom, rx, sondes in (
    ("etoile du mot vide", etoile(EPS), ["", "a", "ab"]),
    ("etoile imbriquee", etoile(etoile(lit("a"))), ["", "a", "aa", "b"]),
):
    motif = rendre(rx)
    regle = re.compile(motif)          # leve re.error si la chaine n'est pas un motif valide
    acceptes = [m for m in sondes if regle.fullmatch(m)]
    print(f"{nom:<20} motif={motif!r:<9} compile=oui  accepte={acceptes!r}")

(a|b)*abb  noeuds=10  rendu='(a|b)*abb'
a(b|c)*    noeuds= 6  rendu='a(b|c)*'
(ab)*      noeuds= 4  rendu='(ab)*'
etoile du mot vide   motif=''        compile=oui  accepte=['']
etoile imbriquee     motif='(a*)*'   compile=oui  accepte=['', 'a', 'aa']


**Lecture.** Les trois arbres se rendent exactement comme leurs notations usuelles :
`(a|b)*abb`, `a(b|c)*`, `(ab)*` — dix, six et quatre nœuds. Les parenthèses n'apparaissent
que là où la priorité l'exige, ce qui compte pour la suite : c'est cette même chaîne que le
moteur externe lira, et un rendu faux ferait diverger l'oracle sans que nos automates
soient en cause.

Le compteur de nœuds est là pour une raison précise : la construction de la section
suivante crée **deux états par nœud**, donc il annonce la taille de l'automate avant
même de le construire.


## 2. De l'expression à l'automate : induction structurelle

Le théorème 3.7 se démontre par **induction structurelle** sur l'arbre, avec trois cas de
base et trois cas d'induction. L'énoncé du manuel impose trois invariants au fragment
construit pour chaque sous-expression : un **seul** état final, aucune flèche qui entre
par l'état initial, aucune flèche qui sort de l'état final. Ce sont eux qui rendent la
composition possible sans conflit.

| Cas | Fragment | États créés |
|---|---|---|
| base $\varnothing$ | entrée et sortie isolées, aucun chemin | 2 |
| base $\varepsilon$ | une epsilon-flèche entrée → sortie | 2 |
| base $a$ | une flèche étiquetée $a$ | 2 |
| $r \mid s$ | nouvelle entrée branchée sur les deux fragments, nouvelles flèches vers la nouvelle sortie | 2 |
| $rs$ | les deux fragments en série, epsilon-flèche de la sortie du premier vers l'entrée du second | 2 |
| $r^\star$ | boucle epsilon de la sortie vers l'entrée, plus un saut epsilon entrée → sortie | 2 |

Chaque nœud consomme donc exactement deux états neufs : le nombre d'états est
$2 \times$ (nombre de nœuds). Les epsilon-transitions ne sont pas un détail
d'implémentation ici — c'est la construction elle-même qui les produit.


In [2]:
class Thompson:
    """Expression -> epsilon-AFN, par induction structurelle (theoreme 3.7 du manuel).

    Chaque noeud de l'AST cree deux etats neufs : une entree et une sortie. Le fragment
    d'un noeud a une seule entree, une seule sortie, aucune fleche entrante sur son
    entree ni sortante de sa sortie : les trois invariants de la preuve, verifies plus bas.
    """

    def __init__(self) -> None:
        self.delta: dict = {}   # (etat, lettre) -> ensemble d'etats
        self.eps: dict = {}     # etat -> ensemble d'etats atteignables sans lire
        self.suivant = 0        # compteur d'etats crees

    def _neuf(self) -> int:
        """Un etat neuf, numerote dans l'ordre de creation."""
        self.suivant += 1
        return self.suivant - 1

    @staticmethod
    def _fleche(table: dict, cle, cible: int) -> None:
        """Ajoute une fleche, en accumulant les cibles quand la cle existe deja."""
        table.setdefault(cle, set()).add(cible)

    def fragment(self, rx):
        """Renvoie (entree, sortie) du fragment qui reconnait exactement L(rx)."""
        if rx is None:                        # base : langage vide, aucun chemin possible
            return self._neuf(), self._neuf()
        if rx == EPS:                         # base : epsilon, une fleche vide
            e, s = self._neuf(), self._neuf()
            self._fleche(self.eps, e, s)
            return e, s
        if rx[0] == "l":                      # base : une lettre, une fleche etiquetee
            e, s = self._neuf(), self._neuf()
            self._fleche(self.delta, (e, rx[1]), s)
            return e, s
        e, s = self._neuf(), self._neuf()
        if rx[0] == "u":                      # induction : deux branches en parallele
            for branche in rx[1:]:
                a, b = self.fragment(branche)
                self._fleche(self.eps, e, a)
                self._fleche(self.eps, b, s)
        elif rx[0] == "c":                    # induction : deux fragments en serie
            a, b = self.fragment(rx[1])
            c, d = self.fragment(rx[2])
            self._fleche(self.eps, e, a)
            self._fleche(self.eps, b, c)
            self._fleche(self.eps, d, s)
        else:                                 # induction : etoile, boucle plus saut direct
            a, b = self.fragment(rx[1])
            self._fleche(self.eps, e, a)
            self._fleche(self.eps, b, a)
            self._fleche(self.eps, b, s)
            self._fleche(self.eps, e, s)
        return e, s

    def construire(self, rx):
        """AFN complet : (etats, alphabet, delta, eps, initial, finaux)."""
        initial, sortie = self.fragment(rx)
        return set(range(self.suivant)), lettres(rx), self.delta, self.eps, initial, {sortie}


def lettres(rx) -> set:
    """Alphabet effectif de l'expression : les lettres qui y apparaissent."""
    if rx is None or rx[0] == "e":
        return set()
    if rx[0] == "l":
        return {rx[1]}
    return set().union(*(lettres(sous) for sous in rx[1:]))


def fermeture(S, eps: dict) -> set:
    """Etats atteignables depuis S par epsilon-transitions seules."""
    vus, pile = set(S), list(S)
    while pile:
        q = pile.pop()
        for r in eps.get(q, ()):
            if r not in vus:
                vus.add(r)
                pile.append(r)
    return vus


def accepte(afn, mot: str) -> bool:
    """True ssi un chemin de lecture du mot finit dans un etat final."""
    etats, alphabet, delta, eps, initial, finaux = afn
    courant = fermeture({initial}, eps)
    for lettre in mot:
        suivant = set()
        for q in courant:
            suivant |= delta.get((q, lettre), set())
        courant = fermeture(suivant, eps)
        if not courant:
            return False
    return bool(courant & finaux)


def entrantes(afn, etat: int) -> int:
    """Nombre de fleches qui arrivent sur cet etat (etiquetees ou epsilon)."""
    _, _, delta, eps, _, _ = afn
    return sum(etat in cibles for cibles in delta.values()) + sum(
        etat in cibles for cibles in eps.values()
    )


def sortantes(afn, etat: int) -> int:
    """Nombre de fleches qui partent de cet etat (etiquetees ou epsilon)."""
    _, _, delta, eps, _, _ = afn
    return sum(1 for (q, _lettre) in delta if q == etat) + len(eps.get(etat, ()))


afn_abb = Thompson().construire(RX_ABB)
etats, alphabet, delta, eps, initial, finaux = afn_abb
(final,) = finaux

# Les trois invariants du theoreme 3.7, mesures sur l'automate construit.
print(f"(a|b)*abb : {len(etats)} etats pour {nb_noeuds(RX_ABB)} noeuds "
      f"-> deux par noeud : {len(etats) == 2 * nb_noeuds(RX_ABB)}")
print(f"un seul etat final : {len(finaux) == 1} ; "
      f"fleches entrantes sur l'initial : {entrantes(afn_abb, initial)} ; "
      f"fleches sortantes du final : {sortantes(afn_abb, final)}")
print(f"arcs etiquetes : {sum(len(v) for v in delta.values())} ; "
      f"epsilon-arcs : {sum(len(v) for v in eps.values())}")

# Quelques mots, pour voir la machine tourner avant de la verifier exhaustivement.
for mot in ("", "abb", "aabb", "ab", "babb"):
    print(f"  {mot!r:8} accepte={accepte(afn_abb, mot)}")

(a|b)*abb : 20 etats pour 10 noeuds -> deux par noeud : True
un seul etat final : True ; fleches entrantes sur l'initial : 0 ; fleches sortantes du final : 0
arcs etiquetes : 5 ; epsilon-arcs : 17
  ''       accepte=False
  'abb'    accepte=True
  'aabb'   accepte=True
  'ab'     accepte=False
  'babb'   accepte=True


**Lecture.** Vingt états pour dix nœuds, et les trois invariants tiennent : un seul état
final, aucune flèche entrante sur l'initial, aucune sortante du final. Les dix-sept flèches
vides se répartissent comme la construction le prévoit — quatre pour l'étoile, quatre pour
l'union (deux par branche), trois par concaténation — contre cinq arcs étiquetés, un par
lettre de l'expression. L'automate est donc surtout fait d'epsilon-transitions.

Vingt états pour un langage qu'un AFD à quatre états suffit à reconnaître : la construction
de Thompson est **simple à démontrer**, pas économique. C'est le prix de l'induction
structurelle, et la section 4 le mesure.

`"babb"` est accepté comme `"abb"` : l'étoile initiale absorbe le préfixe. `"ab"` ne l'est
pas. Rien de surprenant — mais cinq mots ne prouvent rien, d'où la vérification qui suit.


In [3]:
import re


def cle(S) -> str:
    """Cle canonique d'un ensemble d'etats : triee, jointe par des virgules."""
    return ",".join(str(q) for q in sorted(S))


def determiniser(afn):
    """Construction des sous-ensembles ; un etat de l'AFD est nomme par sa cle triee.

    Nommer les etats par une chaine canonique (et non par un frozenset) rend l'automate
    lisible, et rend surtout l'ordre d'elimination de la section 3 reproductible.
    """
    etats, alphabet, delta, eps, initial, finaux = afn
    depart = frozenset(fermeture({initial}, eps))
    nom = {depart: cle(depart)}
    transitions, a_traiter = {}, [depart]
    while a_traiter:
        courant = a_traiter.pop()
        for lettre in sorted(alphabet):
            suivant = set()
            for q in courant:
                suivant |= delta.get((q, lettre), set())
            cible = frozenset(fermeture(suivant, eps))
            if cible not in nom:
                nom[cible] = cle(cible)
                a_traiter.append(cible)
            transitions[(nom[courant], lettre)] = nom[cible]
    return (
        set(nom.values()), alphabet, transitions, nom[depart],
        {nom[S] for S in nom if S & finaux},
    )


def afd_accepte(afd, mot: str) -> bool:
    """Lecture deterministe : une seule transition par lettre."""
    etats, alphabet, delta, initial, finaux = afd
    courant = initial
    for lettre in mot:
        courant = delta[(courant, lettre)]
    return courant in finaux


def mots(alphabet: str, n_max: int):
    """Tous les mots de longueur 0 a n_max sur l'alphabet, mot vide compris."""
    for n in range(n_max + 1):
        for t in product(alphabet, repeat=n):
            yield "".join(t)


# L'AFN a quatre etats du carnet precedent, pour le meme langage : les mots finissant par abb.
AFN_QUATRE = (
    {0, 1, 2, 3}, {"a", "b"},
    {(0, "a"): {0, 1}, (0, "b"): {0}, (1, "b"): {2}, (2, "b"): {3}},
    {}, 0, {3},
)

# Une seconde expression, de forme differente : etoile des deux cotes du motif.
RX_ABA = concat(etoile(AB), concat(lit("a"), concat(lit("b"), concat(lit("a"), etoile(AB)))))

tous = list(mots("ab", 9))
cas = [
    ("(a|b)*abb", RX_ABB, afn_abb, AFN_QUATRE),
    ("(a|b)*aba(a|b)*", RX_ABA, Thompson().construire(RX_ABA), None),
]
for nom, rx, afn, main in cas:
    motif = rendre(rx)                       # l'oracle lit la meme chaine que celle affichee
    afd = determiniser(afn)
    attendu = [bool(re.fullmatch(motif, m)) for m in tous]
    obtenu = [accepte(afn, m) for m in tous]
    par_afd = [afd_accepte(afd, m) for m in tous]
    print(f"{nom:<16} AFN {len(afn[0]):2d} etats -> AFD {len(afd[0]):2d} etats ; "
          f"acceptes {sum(obtenu):4d}/{len(tous)} ; "
          f"divergences AFN/re : {sum(a != o for a, o in zip(attendu, obtenu))} ; "
          f"AFN/AFD : {sum(a != o for a, o in zip(par_afd, obtenu))}")
    if main is not None:                     # en plus, l'AFN ecrit a la main dans Automata-01
        quatre = [accepte(main, m) for m in tous]
        print(f"{'  (AFN 4 etats)':<16} acceptes {sum(quatre):4d}/{len(tous)} ; "
              f"divergences contre re : {sum(a != b for a, b in zip(quatre, attendu))}")


def classes_equivalence(afd) -> int:
    """Nombre de classes d'etats equivalents : meme statut final, memes transitions.

    Ce compteur dit si l'automate construit est minimal, sans construire l'automate
    quotient -- c'est le role de Hopcroft dans Automata-01. Le raffinement s'arrete
    quand la partition ne se divise plus.
    """
    etats, alphabet, delta, initial, finaux = afd
    classe = {e: e in finaux for e in etats}     # partition de depart : final ou non
    while True:
        signature = {
            e: (classe[e], tuple(classe[delta[(e, lettre)]] for lettre in sorted(alphabet)))
            for e in etats
        }
        if len(set(signature.values())) == len(set(classe.values())):
            return len(set(signature.values()))
        classe = signature


print()
print("Etats de l'AFD construit, puis classes d'etats equivalents :")
for nom, rx in (("(a|b)*abb", RX_ABB), ("(a|b)*aba(a|b)*", RX_ABA),
                ("a(b|c)*", RX_ABC), ("(ab)*", RX_AB_STAR)):
    afd = determiniser(Thompson().construire(rx))
    print(f"{nom:<16} {len(afd[0]):2d} etats -> {classes_equivalence(afd)} classe(s)")

(a|b)*abb        AFN 20 etats -> AFD  5 etats ; acceptes  127/1023 ; divergences AFN/re : 0 ; AFN/AFD : 0
  (AFN 4 etats)  acceptes  127/1023 ; divergences contre re : 0
(a|b)*aba(a|b)*  AFN 30 etats -> AFD  9 etats ; acceptes  560/1023 ; divergences AFN/re : 0 ; AFN/AFD : 0

Etats de l'AFD construit, puis classes d'etats equivalents :
(a|b)*abb         5 etats -> 4 classe(s)
(a|b)*aba(a|b)*   9 etats -> 4 classe(s)
a(b|c)*           5 etats -> 3 classe(s)
(ab)*             4 etats -> 3 classe(s)


**Lecture.** Quatre lectures d'un même langage — l'AFN de Thompson, l'AFD obtenu par
sous-ensembles, un `re.fullmatch` sur la chaîne rendue, et l'AFN à quatre états écrit à la
main dans `Automata-01` — donnent le même verdict sur chacun des 1023 mots de longueur au
plus 9, sans une divergence.

Le décompte des mots acceptés pour `(a|b)*abb` n'est pas un hasard : sur les 1023 mots,
ceux qui finissent par `abb` sont au nombre de 127 (un mot par préfixe libre, de longueur
0 à 6). Pour `(a|b)*aba(a|b)*`, 560 mots contiennent le facteur `aba`.

Le second tableau est le plus instructif : l'AFD obtenu compte **cinq** états là où le plus
petit en demande quatre, et cinq là où `a(b|c)*` n'en demande que trois. Fusionner les états
équivalents — même statut final, mêmes transitions — ramène 5 à 4, puis 5 à 3 : l'écart
n'est donc pas toujours d'un seul état. La construction par sous-ensembles garantit le
**langage**, jamais la **minimalité** ; c'est le travail de `Hopcroft` dans `Automata-01`,
et c'est aussi pourquoi l'aller-retour du paragraphe 3 revient avec des expressions plus
lourdes que celles d'où il part.


## 3. De l'automate à l'expression : éliminer les états

Le sens inverse est le théorème 3.4, et le manuel en donne **d'abord** une preuve
récursive (§3.2.1 : des expressions $R_{ij}^{(k)}$ qui décrivent les chemins passant par
les états $1..k$), puis une **variante plus économique** au §3.2.2, celle qu'on implémente
ici : *éliminer les états*.

L'idée tient en une phrase, citée du manuel : quand on élimine un état $s$, « les chemins
qui passaient par $s$ n'existent plus dans l'automate ; pour que le langage ne change
pas, il faut ajouter sur un arc allant directement de $q$ à $p$ les étiquettes des chemins
qui allaient de $q$ à $p$ en passant par $s$ ». Comme ces étiquettes sont des ensembles
de mots — parfois infinis — on les écrit comme des expressions régulières, et l'automate
devient un automate **étiqueté par des expressions**.

Pour éliminer $s$, on remplace donc chaque paire (prédécesseur $q_i$, successeur $p_j$)
par l'arc $q_i \to p_j$ étiqueté

$$R(q_i, s)\; R(s, s)^\star\; R(s, p_j)$$

et l'on retire tous les arcs touchant $s$. Quand il ne reste plus que deux nœuds, la
réponse est l'étiquette de l'arc qui les relie. Deux états synthétiques — une source
reliée à l'état initial, un puits relié aux états finaux — ramènent l'automate à un seul
point d'entrée et un seul point de sortie.

Le manuel chiffre le coût des deux méthodes : pour la récursion du §3.2.1, « la longueur
de l'expression peut croître d'un facteur 4 en moyenne à chaque étape d'induction, si bien
que les expressions elles-mêmes peuvent atteindre l'ordre de $4^n$ symboles ». L'élimination
retarde la duplication, elle ne la supprime pas — et **l'ordre dans lequel on élimine les
états change la taille du résultat**, jamais son langage.


In [4]:
def U(x, y):
    """Union algebrique : le langage vide est neutre, les doublons disparaissent."""
    if x is None:
        return y
    if y is None or x == y:
        return x
    return ("u", x, y)


def C(x, y):
    """Concatenation algebrique : le langage vide est absorbant, epsilon est neutre."""
    if x is None or y is None:
        return None
    if x == EPS:
        return y
    if y == EPS:
        return x
    return ("c", x, y)


def E(x):
    """Etoile algebrique : vide* = epsilon* = epsilon, et (r*)* = r*."""
    if x is None or x == EPS or x[0] == "s":
        return EPS
    return ("s", x)


def ordre_etats(etat) -> tuple:
    """Ordre d'elimination : noms courts d'abord, puis ordre lexicographique."""
    return (len(str(etat)), str(etat))


def regex_depuis_afd(afd, ordre=None):
    """Elimination d'etats (manuel, section 3.2.2) : AST d'une expression du langage de l'AFD."""
    etats, alphabet, delta, initial, finaux = afd
    arcs = {}                                   # (i, j) -> expression etiquetant l'arc
    for (p, lettre), q in delta.items():
        arcs[(p, q)] = U(arcs.get((p, q)), lit(lettre))
    source, puits = "source", "puits"           # etats synthetiques : une entree, une sortie
    arcs[(source, initial)] = U(arcs.get((source, initial)), EPS)
    for f in sorted(finaux, key=ordre_etats):
        arcs[(f, puits)] = U(arcs.get((f, puits)), EPS)
    vivants = set(etats) | {source, puits}
    sequence = sorted(etats, key=ordre_etats) if ordre is None else ordre
    for k in sequence:
        boucle = E(arcs.get((k, k)))            # R(k, k)*, epsilon si pas de boucle
        entrants = [i for i in sorted(vivants, key=ordre_etats) if i != k and (i, k) in arcs]
        sortants = [j for j in sorted(vivants, key=ordre_etats) if j != k and (k, j) in arcs]
        for i in entrants:
            for j in sortants:
                detour = C(C(arcs[(i, k)], boucle), arcs[(k, j)])
                arcs[(i, j)] = U(arcs.get((i, j)), detour)
        for arete in [a for a in arcs if k in a]:
            del arcs[arete]                     # tous les arcs qui touchaient k disparaissent
        vivants.discard(k)
    return arcs.get((source, puits))


# Premier AFD, ecrit a la main : nombre pair de 1 sur {0, 1} (etat 0 = pair).
PARITE = (
    {0, 1}, {"0", "1"},
    {(0, "0"): 0, (0, "1"): 1, (1, "0"): 1, (1, "1"): 0},
    0, {0},
)
rx_parite = regex_depuis_afd(PARITE)
print(f"parite du nombre de 1 : {rendre(rx_parite)!r}  ({nb_noeuds(rx_parite)} noeuds)")

# Deuxieme AFD : les mots qui contiennent le facteur 01.
CONTIENT_01 = (
    {0, 1, 2}, {"0", "1"},
    {(0, "0"): 1, (0, "1"): 0, (1, "0"): 1, (1, "1"): 2, (2, "0"): 2, (2, "1"): 2},
    0, {2},
)
rx_01 = regex_depuis_afd(CONTIENT_01)
print(f"contient 01 : {rendre(rx_01)!r}  ({nb_noeuds(rx_01)} noeuds)")

# Verification : l'expression produite et l'AFD de depart doivent accepter les memes mots.
tous_01 = list(mots("01", 10))
for nom, afd, rx in (("parite", PARITE, rx_parite), ("contient 01", CONTIENT_01, rx_01)):
    motif = rendre(rx)
    divergences = sum(
        afd_accepte(afd, m) != bool(re.fullmatch(motif, m)) for m in tous_01
    )
    print(f"{nom:<12} {len(tous_01)} mots jusqu'a la longueur 10 : "
          f"{divergences} divergence(s) contre re.fullmatch")

parite du nombre de 1 : '0*|0*1(0|10*1)*10*'  (22 noeuds)
contient 01 : '1*00*1(0|1)*'  (14 noeuds)
parite       2047 mots jusqu'a la longueur 10 : 0 divergence(s) contre re.fullmatch
contient 01  2047 mots jusqu'a la longueur 10 : 0 divergence(s) contre re.fullmatch


**Lecture.** L'élimination redonne `0*|0*1(0|10*1)*10*` pour la parité du nombre de `1`
(22 nœuds). L'expression est exacte — aucune divergence sur les 2047 mots de longueur au
plus 10 — et se relit pourtant : ou bien `0*`, aucun `1` du tout ; ou bien un premier `1`,
puis des blocs qui ajoutent chacun zéro ou deux `1` (`0`, ou `10*1`), puis un dernier `1`
et des zéros. Le compte est pair dans les deux cas.

Le second AFD donne `1*00*1(0|1)*` (14 nœuds), qui se lit d'un trait : « des `1`, un `0`
avec éventuellement d'autres `0`, un `1`, puis n'importe quoi » — c'est bien « contient le
facteur `01` ». Douze caractères contre huit pour `(0|1)*01`, qui décrit le même langage.

L'élimination ne cherche pas la plus petite expression. Elle en cherche une **correcte**,
et rien dans le procédé ne garantit que le résultat soit court.


In [5]:
# Aller-retour complet : expression -> AFN (Thompson) -> AFD (sous-ensembles)
# -> expression (elimination). Le theoreme annonce le meme langage au depart et a l'arrivee ;
# on le verifie sur tous les mots jusqu'a la longueur 7, avec re comme arbitre externe.
for nom, rx in (("(a|b)*abb", RX_ABB), ("a(b|c)*", RX_ABC), ("(ab)*", RX_AB_STAR)):
    afn = Thompson().construire(rx)
    afd = determiniser(afn)
    retour = regex_depuis_afd(afd)
    alphabet_essai = "".join(sorted(lettres(rx)))
    essai = list(mots(alphabet_essai, 7))
    depart = [accepte(afn, m) for m in essai]
    arrivee = [bool(re.fullmatch(rendre(retour), m)) for m in essai]
    print(f"{nom:<10} AFD {len(afd[0]):2d} etats ; retour {rendre(retour)!r} "
          f"({nb_noeuds(retour)} noeuds) ; {len(essai)} mots, "
          f"{sum(a != b for a, b in zip(depart, arrivee))} divergence(s)")

(a|b)*abb  AFD  5 etats ; retour '(a|bb*a)(a|b(a|b(a|bb*a)))*bb' (30 noeuds) ; 255 mots, 0 divergence(s)
a(b|c)*    AFD  5 etats ; retour 'a|abb*|(ac|abb*c)(c|bb*c)*(|bb*)' (38 noeuds) ; 3280 mots, 0 divergence(s)
(ab)*      AFD  4 etats ; retour '|ab(ab)*' (10 noeuds) ; 255 mots, 0 divergence(s)


**Lecture.** Trois allers-retours, aucune divergence : la composition des deux théorèmes
rend bien une expression du **même langage**, jamais la même chaîne.

- `(a|b)*abb` : 10 nœuds au départ, 30 à l'arrivée, 255 mots vérifiés ;
- `a(b|c)*` : 6 nœuds au départ, 38 à l'arrivée, 3280 mots vérifiés ;
- `(ab)*` : 4 nœuds au départ, 10 à l'arrivée, 255 mots vérifiés.

Les trois reviennent plus lourdes, et le passage par un AFD non minimal y est pour
quelque chose : chaque état de trop est un état de plus où l'élimination duplique des
étiquettes. Le dernier cas montre aussi comment $\varepsilon$ s'écrit dans ce rendu : une
alternative vide en tête, `|ab(ab)*`, que `re` lit comme « le mot vide, ou `ab` répété ».


## 4. Ce que coûte la conversion

Les trois conversions n'ont pas le même régime de coût, et c'est ce qui décide de leur
usage :

| Conversion | Majorant | Pourquoi |
|---|---|---|
| expression → AFN | **linéaire** : deux états par nœud | chaque nœud ajoute deux états, jamais plus |
| AFN → AFD | jusqu'à $2^n$ pour $n$ états | un état de l'AFD est un *ensemble* d'états de l'AFN |
| AFD → expression | peut exploser (le manuel cite $4^n$ pour la récursion) | chaque élimination duplique les étiquettes |

Plutôt que de réciter ces majorants, mesurons-les sur deux familles paramétrées. La
première est la famille classique $(a|b)^\star a\,(a|b)^n$, dont l'AFD minimal a
$2^{n+1}$ états. La seconde est la famille « le mot finit par $a^n$ », dont l'AFD a
$n+1$ états : on lui applique l'élimination dans deux ordres différents.


In [6]:
def afd_finit_par_a(n: int):
    """AFD a n+1 etats : les mots sur {a, b} dont le suffixe est a^n."""
    delta = {}
    for i in range(n + 1):
        delta[(i, "a")] = min(i + 1, n)     # on avance dans la serie de a, puis on sature
        delta[(i, "b")] = 0                 # un b casse la serie : retour a l'etat initial
    return set(range(n + 1)), {"a", "b"}, delta, 0, {n}


print("Determinisation : (a|b)*a(a|b)^n, dont l'AFD minimal a 2^(n+1) etats")
print(f"{'n':>2} {'etats AFN':>10} {'etats AFD':>10} {'apres fusion':>13}")
for n in range(1, 9):
    rx = concat(etoile(AB), lit("a"))
    for _ in range(n):
        rx = concat(rx, AB)
    afn = Thompson().construire(rx)
    afd = determiniser(afn)
    print(f"{n:>2} {len(afn[0]):>10} {len(afd[0]):>10} {classes_equivalence(afd):>13}")

print()
print("Elimination : famille 'suffixe a^n' (n+1 etats), ordre naturel puis ordre inverse")
print(f"{'n':>2} {'noeuds':>7} {'caracteres':>11} {'noeuds inv.':>12} {'car. inv.':>10}")
for n in range(1, 7):
    afd = afd_finit_par_a(n)
    naturel = regex_depuis_afd(afd)
    inverse = regex_depuis_afd(afd, ordre=sorted(afd[0], key=ordre_etats, reverse=True))
    print(f"{n:>2} {nb_noeuds(naturel):>7} {len(rendre(naturel)):>11} "
          f"{nb_noeuds(inverse):>12} {len(rendre(inverse)):>10}")
    if n == 4:
        # Les deux ordres doivent donner le meme langage : on le verifie au passage.
        essai = list(mots("ab", 8))
        ecart = sum(
            bool(re.fullmatch(rendre(naturel), m)) != bool(re.fullmatch(rendre(inverse), m))
            for m in essai
        )
        print(f"   n=4 : les deux ordres divergent sur {ecart} des {len(essai)} mots testes")

Determinisation : (a|b)*a(a|b)^n, dont l'AFD minimal a 2^(n+1) etats
 n  etats AFN  etats AFD  apres fusion
 1         20          5             4
 2         28          9             8
 3         36         17            16
 4         44         33            32
 5         52         65            64
 6         60        129           128
 7         68        257           256


 8         76        513           512

Elimination : famille 'suffixe a^n' (n+1 etats), ordre naturel puis ordre inverse
 n  noeuds  caracteres  noeuds inv.  car. inv.
 1      14          12           14         12
 2      34          28           20         18
 3      74          60           26         24
 4     154         124           32         30
   n=4 : les deux ordres divergent sur 0 des 511 mots testes
 5     314         252           38         36
 6     634         508           44         42


**Lecture.** La déterminisation **double** d'un pas à l'autre : 5, 9, 17, 33, …, 513
états pour l'AFD, quand l'AFN ne gagne que huit états par pas (20, 28, 36, …, 76). Ce
doublement est l'exponentielle du tableau, mesurée et non récitée. La dernière colonne
montre aussi que l'automate construit n'est pas minimal : une fois les états équivalents
fusionnés, il reste exactement $2^{n+1}$ états — **un** état de trop à chaque pas, jamais
davantage.

Sur la seconde famille, les deux ordres d'élimination ne coûtent pas du tout la même
chose. Dans l'ordre **inverse** — l'état final d'abord — la taille croît **linéairement** :
14, 20, 26, 32, 38, 44 nœuds, six de plus par pas. Dans l'ordre **naturel** — l'état
initial d'abord — elle double à peu près à chaque pas : 14, 34, 74, 154, 314, 634 nœuds.
Les deux expressions décrivent pourtant le même langage : vérifié sur 511 mots au cas
$n = 4$, zéro divergence.

L'explication tient à ce que l'élimination transporte. Dans cet AFD, **toutes** les flèches
étiquetées `b` reviennent à l'état 0 : éliminer 0 en premier oblige à recopier sa boucle
dans les arcs qui partent des autres états, et chaque élimination suivante duplique à
nouveau ces étiquettes. Éliminer l'état final en premier fait au contraire replier la
chaîne des `a` sur un seul arc, dont l'étiquette gagne une lettre par pas — d'où la
croissance linéaire.

À retenir pour la conception : écrire une expression coûte peu, la déterminiser peut coûter
cher, et revenir à une expression depuis un automate coûte ce que coûte l'ordre choisi.


## 5. Ce que le dépôt fait déjà avec les expressions régulières

Trois carnets voisins traitent la même matière par d'autres routes. Les renvois sont
précis, pour que le lecteur sache quoi y chercher.

- **`SymbolicAI/SMT/Z3-API/Z3-04-Strings-Regex-*`**, sections 3 à 5 : les expressions
  régulières y sont des **objets symboliques** du solveur (`Re`, `InRe`). La différence
  avec ce carnet est celle de l'usage : ici l'expression est un **programme** qu'on exécute,
  là-bas c'est une **contrainte** que le solveur décide — et dont il peut exhiber un mot
  témoin. C'est le pont naturel vers les questions de la forme « existe-t-il un mot qui
  satisfait ces deux motifs à la fois ? ».
- **`Search/Part1-Foundations/Search-10-SymbolicAutomata`**, sections 3 et 4 : les
  transitions y sont étiquetées par des **prédicats** Z3 (intervalles d'entiers), ce qui
  autorise un alphabet infini. La construction de Thompson suppose un alphabet fini et
  énumérable : c'est exactement l'hypothèse que les automates symboliques relâchent.
- **`SymbolicAI/Lean/Lean-14-Finiteness-Derivatives`** : les **dérivées de Brzozowski**
  construisent l'automate *directement depuis l'arbre*, sans passer par les
  epsilon-transitions. Le carnet formalise la finitude de l'ensemble des dérivées, qui est
  ce qui garantit un matching linéaire. C'est la troisième route entre expression et
  automate, après Thompson (section 2) et l'élimination (section 3).

## Bilan

- Une expression régulière se convertit en automate par **induction structurelle** : trois
  cas de base, trois cas de composition, deux états par nœud, et des epsilon-transitions
  qui viennent de la construction elle-même.
- Un automate se convertit en expression par **élimination d'états** : la boucle d'un état
  devient une étoile, chaque paire (prédécesseur, successeur) reçoit l'étiquette du détour,
  et l'ordre d'élimination arbitre entre taille et lisibilité.
- Les deux sens se composent, et l'aller-retour revient sur le **même langage** — vérifié
  mot par mot contre un moteur indépendant.
- Le coût n'est pas symétrique : linéaire dans un sens, potentiellement exponentiel dans
  l'autre. Une expression courte peut cacher un automate énorme.

Ce que ce carnet ne fait pas : décider si un langage **est** régulier. C'est l'objet du
lemme de pompage, déjà abordé en exercice dans `Automata-01`, et le point de départ de la
sous-série : les automates à pile (`Automata-03`) quittent la classe des langages réguliers.


## Exercices

Trois exercices, dans l'ordre du carnet. Chaque cellule s'exécute **sans erreur telle
qu'elle est livrée** : remplacez les lignes `# TODO etudiant` par votre solution, puis
relancez la cellule.


### Exercice 1 — Construire l'automate d'une expression

Construisez l'AST de l'expression $(ab \mid c)^\star a$ avec les constructeurs du
paragraphe 1 (`lit`, `union`, `concat`, `etoile`), puis laissez la cellule vérifier le
langage obtenu contre `re.fullmatch(r"(ab|c)*a", ...)` sur tous les mots de longueur au
plus 7 sur $\{a, b, c\}$.

*(Indice : `ab` est une concaténation de deux littéraux, `(ab|c)` une union, et l'étoile
porte sur cette union avant la concaténation finale avec `a`.)*


In [7]:
# TODO etudiant : construire l'AST de (ab|c)*a (remplacez None).
rx_ex1 = None

# Verification : a executer apres votre construction, sans la modifier.
if rx_ex1 is not None:
    afn_ex1 = Thompson().construire(rx_ex1)
    essai = list(mots("abc", 7))
    divergences = sum(
        accepte(afn_ex1, m) != bool(re.fullmatch(r"(ab|c)*a", m)) for m in essai
    )
    print(f"{len(essai)} mots testes ; divergences : {divergences} (attendu 0)")
else:
    print("Exercice a completer : construisez rx_ex1 (un AST) puis relancez cette cellule.")

Exercice a completer : construisez rx_ex1 (un AST) puis relancez cette cellule.


### Exercice 2 — Éliminer les états à la main

Voici un AFD à trois états sur $\{0, 1\}$ qui accepte les mots se terminant par `01` :

| État | sur `0` | sur `1` |
|---|---|---|
| 0 (initial) | 1 | 0 |
| 1 | 1 | 2 |
| 2 (final) | 1 | 0 |

Appliquez l'élimination d'états **sur papier**, puis écrivez le résultat comme une chaîne
Python lisible par `re.fullmatch`. La cellule vérifie votre chaîne contre l'AFD sur tous
les mots de longueur au plus 8.


In [8]:
AFD_EX2 = (
    {0, 1, 2}, {"0", "1"},
    {(0, "0"): 1, (0, "1"): 0, (1, "0"): 1, (1, "1"): 2, (2, "0"): 1, (2, "1"): 0},
    0, {2},
)

# TODO etudiant : ecrire ici l'expression trouvee a la main (chaine Python, remplacez None).
chaine_ex2 = None

# Verification : a executer apres votre derivation, sans la modifier.
if chaine_ex2 is not None:
    essai = list(mots("01", 8))
    divergences = sum(
        afd_accepte(AFD_EX2, m) != bool(re.fullmatch(chaine_ex2, m)) for m in essai
    )
    print(f"{len(essai)} mots testes ; divergences : {divergences} (attendu 0)")
    print(f"Votre expression : {chaine_ex2!r} ; "
          f"celle de l'organe du paragraphe 3 : {rendre(regex_depuis_afd(AFD_EX2))!r}")
else:
    print("Exercice a completer : chaine_ex2 = \"...\"  # votre expression")
    print("Indication : re.fullmatch exige que l'expression decrive le mot ENTIER,")
    print("du premier caractere au dernier ; `01` seul ne reconnait que le mot 01.")

Exercice a completer : chaine_ex2 = "..."  # votre expression
Indication : re.fullmatch exige que l'expression decrive le mot ENTIER,
du premier caractere au dernier ; `01` seul ne reconnait que le mot 01.


### Exercice 3 — Pourquoi l'ordre d'élimination compte

Dans le tableau du paragraphe 4, la famille « suffixe $a^n$ » donne une expression dont
la taille croît **linéairement** quand on élimine les états du dernier au premier, et
**double à peu près à chaque pas** quand on les élimine du premier au dernier.

**Expliquez en trois phrases** d'où vient cette différence, en vous appuyant sur la règle
d'élimination $R(q_i, s)\,R(s, s)^\star\,R(s, p_j)$ : quels arcs sont dupliqués, et
pourquoi l'ordre décide de la taille des étiquettes qu'on transporte. *(Indice : dans cet
AFD, toutes les flèches étiquetées `b` reviennent à l'état 0. Regardez ce qui se recopie
sur les autres arcs quand on élimine 0 en premier, et ce qui se passe au contraire quand
on élimine l'état final en premier.)*


In [9]:
# Reponse a rediger en commentaire (exercice de raisonnement).
# TODO etudiant : quand on elimine l'etat ...
# ... donc l'expression naturelle ...
print("Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.")

Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.
